In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, mean_absolute_error

import joblib


In [17]:
file_path = "filtered_yelp_review.json"

print("Loading data...")
df = pd.read_json(file_path, lines=True, nrows=500000)

print("Loaded rows:", len(df))
df.head()


Loading data...
Loaded rows: 500000


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31


In [18]:
df = df[['text', 'stars']].dropna()

df = df[df['stars'].between(1, 5)]
df['stars'] = df['stars'].astype(int)

print(df['stars'].value_counts().sort_index())
df.head()


stars
1     59270
2     41479
3     56216
4    119876
5    223159
Name: count, dtype: int64


,text,stars
0,"If you decide to eat here, just be aware it is...",3
1,Family diner. Had the buffet. Eclectic assortm...,3
2,"Wow! Yummy, different, delicious. Our favo...",5
3,Cute interior and owner (?) gave us tour of up...,4
4,I am a long term frequent customer of this est...,1


In [19]:
X = df['text']       # input feature = review text
y = df['stars']      # target = rating 1–5

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,         
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Train size: 400000
Test size: 100000


In [20]:
text_rating_clf = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=120000,       
        ngram_range=(1, 3),       
        min_df=3,                 
        sublinear_tf=True,        
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        C=2.0,
        max_iter=2500,
        class_weight='balanced',  
        n_jobs=-1,
        multi_class='multinomial',  
        solver='lbfgs'
    ))
])


In [21]:
print("Training 1–5 rating model...")
text_rating_clf.fit(X_train, y_train)
print("Training complete.")


Training 1–5 rating model...


/Users/vishu/Desktop/412-ML/Hotel-Recommendation-System/hotel_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training complete.


In [22]:
y_pred = text_rating_clf.predict(X_test)

print("Classification report (per rating 1–5):")
print(classification_report(y_test, y_pred))

# Since ratings are ordinal, also compute MAE
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (in stars): {mae:.3f}")


Classification report (per rating 1–5):
              precision    recall  f1-score   support

           1       0.72      0.75      0.73     11854
           2       0.41      0.46      0.43      8296
           3       0.42      0.47      0.45     11243
           4       0.51      0.54      0.52     23975
           5       0.82      0.75      0.78     44632

    accuracy                           0.64    100000
   macro avg       0.58      0.59      0.58    100000
weighted avg       0.66      0.64      0.65    100000

Mean Absolute Error (in stars): 0.423


In [23]:
examples = [
    "The hotel was amazing, super clean and the staff were very friendly.",
    "It was okay, nothing special, not bad but not great either.",
    "Terrible experience. Room was dirty and the service was rude.",
    "Nice location and decent breakfast, but the room was small."
]

pred_ratings = text_rating_clf.predict(examples)
pred_proba = text_rating_clf.predict_proba(examples)

print("Examples:")
for text, rating, probs in zip(examples, pred_ratings, pred_proba):
    print("\nReview:", text)
    print("Predicted rating (1–5):", rating)
    print("Class probabilities (per star):", dict(zip(text_rating_clf.classes_, probs)))


Examples:

Review: The hotel was amazing, super clean and the staff were very friendly.
Predicted rating (1–5): 5
Class probabilities (per star): {np.int64(1): np.float64(9.079245533270159e-05), np.int64(2): np.float64(0.000420105341169952), np.int64(3): np.float64(0.015701875150943907), np.int64(4): np.float64(0.12233459226189604), np.int64(5): np.float64(0.8614526347906573)}

Review: It was okay, nothing special, not bad but not great either.
Predicted rating (1–5): 3
Class probabilities (per star): {np.int64(1): np.float64(0.0010211993306234572), np.int64(2): np.float64(0.02519570004610633), np.int64(3): np.float64(0.9686958209709311), np.int64(4): np.float64(0.004825378449934607), np.int64(5): np.float64(0.00026190120240449387)}

Review: Terrible experience. Room was dirty and the service was rude.
Predicted rating (1–5): 1
Class probabilities (per star): {np.int64(1): np.float64(0.9908476206698125), np.int64(2): np.float64(0.008892449371727632), np.int64(3): np.float64(0.000255265

In [24]:
joblib.dump(text_rating_clf, "sentiment_rating_model_1to5.joblib")
print("Model saved as sentiment_rating_model_1to5.joblib")


Model saved as sentiment_rating_model_1to5.joblib
